# Train the bolt model (Google Colab, free GPU)
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Click the **folder icon** on the left and drag your **dataset.zip** into the panel. Wait until the upload finishes.
3. Click **Runtime → Run all**. Keep this tab open. When training ends, `best.pt` downloads automatically.

In [ ]:
!pip install -q ultralytics
import torch
print("GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then Run all again"

In [ ]:
import zipfile, glob, yaml, os
assert os.path.exists('/content/dataset.zip'), "Upload dataset.zip first (folder icon on the left, then drag the file in)"
zipfile.ZipFile('/content/dataset.zip').extractall('/content/ds')
p = glob.glob('/content/ds/**/data.yaml', recursive=True)[0]
cfg = yaml.safe_load(open(p)); cfg['path'] = os.path.dirname(p)
yaml.safe_dump(cfg, open('/content/data.yaml', 'w'))
print(open('/content/data.yaml').read())

In [ ]:
from ultralytics import YOLO
EPOCHS = 80      # more epochs = better but slower
IMGSZ = 960      # 960 finds small bolts better (use 640 if you get an out-of-memory error)
model = YOLO('yolov8s.pt')
model.train(data='/content/data.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=-1, patience=25, cos_lr=True,
            close_mosaic=10, degrees=10, scale=0.5, hsv_v=0.4, project='/content/runs', name='bolt', exist_ok=True)

In [ ]:
best = '/content/runs/bolt/weights/best.pt'
m = YOLO(best).val(data='/content/data.yaml', imgsz=IMGSZ)
print(f"Precision {m.box.mp*100:.1f}%  Recall {m.box.mr*100:.1f}%  mAP50 {m.box.map50*100:.1f}%")
from google.colab import files
files.download(best)